Let's compare.

### yellowish mask
![text](yellowish_mask_preview.png)


### Test (labels)
![text](m05-label.png)

In [ ]:
Ok, 

# Scan through training image to detect yellow HEX codes.

In [1]:

import cv2
import numpy as np
from collections import Counter

img_path = "m05-label.png"

# ---- Load image safely ----
img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
if img is None:
    raise FileNotFoundError(img_path)

if img.ndim == 2:
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
elif img.shape[2] == 4:
    img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

# ---- Broad yellow-ish filter (HSV) ----
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lower = np.array([10, 40, 80])
upper = np.array([55, 255, 255])
mask = cv2.inRange(hsv, lower, upper) > 0

yellow_pixels = img[mask]
if yellow_pixels.shape[0] == 0:
    raise ValueError("No yellow-ish pixels found; loosen HSV bounds.")

# ---- Quantize colors ----
BIN = 8
q = (yellow_pixels // BIN) * BIN
counts = Counter(map(tuple, q))
top = counts.most_common(15)

def bgr_to_hex(bgr):
    b, g, r = bgr
    return f"#{r:02X}{g:02X}{b:02X}"

hex_list = [bgr_to_hex(bgr) for bgr, _ in top]

# ---- Output (READY TO COPY/PASTE) ----
print("\n--- Python list (copy/paste) ---")
print("TARGET_HEXES = [")
for hx in hex_list:
    print(f'    "{hx}",')
print("]")

print("\n--- Single string (optional) ---")
print(",".join(hex_list))

# Optional preview mask
cv2.imwrite("yellowish_mask_preview.png", (mask.astype(np.uint8) * 255))
print("\nWrote yellowish_mask_preview.png")



--- Python list (copy/paste) ---
TARGET_HEXES = [
    "#E8E800",
    "#E0E000",
    "#C8C808",
    "#D8D808",
    "#D0D008",
    "#F0F000",
    "#888818",
    "#B0B010",
    "#A8A810",
    "#C0C008",
    "#787818",
    "#606020",
    "#A0A010",
    "#808018",
    "#505020",
]

--- Single string (optional) ---
#E8E800,#E0E000,#C8C808,#D8D808,#D0D008,#F0F000,#888818,#B0B010,#A8A810,#C0C008,#787818,#606020,#A0A010,#808018,#505020

Wrote yellowish_mask_preview.png


# Scan through

In [2]:
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

# ----------------------------
# INPUTS
# ----------------------------
img_label = "m05-label.png"
img_nolabel = "m05-nolabel.png"

train_path = Path("train"); train_path.mkdir(exist_ok=True)
test_path  = Path("test");  test_path.mkdir(exist_ok=True)

TILE = 256

# Paste the SAME hex list you exported programmatically
TARGET_HEXES = [
    "#E8E800",
    "#E0E000",
    "#C8C808",
    "#D8D808",
    "#D0D008",
    "#F0F000",
    "#888818",
    "#B0B010",
    "#A8A810",
    "#C0C008",
    "#787818",
    "#606020",
    "#A0A010",
    "#808018",
    "#505020",
]

# Must match the BIN you used when extracting the hexes
BIN = 8

# Tile labeling threshold (thin lines -> keep small)
MIN_MATCH_PIXELS = 3

# ----------------------------
# HELPERS
# ----------------------------
def read_as_bgr(path: str) -> np.ndarray:
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    elif im.ndim == 3 and im.shape[2] == 4:
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    return im

def hex_to_bgr(hex_str: str) -> np.ndarray:
    hex_str = hex_str.lstrip("#")
    r = int(hex_str[0:2], 16)
    g = int(hex_str[2:4], 16)
    b = int(hex_str[4:6], 16)
    return np.array([b, g, r], dtype=np.uint8)

def quantize_bgr(img_bgr: np.ndarray, bin_size: int) -> np.ndarray:
    # Quantize each channel to multiples of BIN
    return (img_bgr // bin_size) * bin_size

def build_mask_from_quantized_palette(img_bgr: np.ndarray, target_hexes: list[str], bin_size: int) -> np.ndarray:
    if not target_hexes:
        raise ValueError("TARGET_HEXES is empty. Paste your exported HEX list.")

    qimg = quantize_bgr(img_bgr, bin_size)  # uint8

    # Quantize the palette the same way
    palette_q = []
    for hx in target_hexes:
        bgr = hex_to_bgr(hx)
        bgr_q = (bgr // bin_size) * bin_size
        palette_q.append(tuple(int(x) for x in bgr_q))
    palette_set = set(palette_q)

    # Vectorized membership test by packing BGR into one int
    pack = (qimg[:, :, 0].astype(np.uint32) << 16) | (qimg[:, :, 1].astype(np.uint32) << 8) | qimg[:, :, 2].astype(np.uint32)
    palette_pack = np.array([(b << 16) | (g << 8) | r for (b, g, r) in palette_set], dtype=np.uint32)

    # membership: for each pixel packed value, is it in palette_pack?
    # Use np.isin (fast enough for single images)
    mask_bool = np.isin(pack, palette_pack)
    return (mask_bool.astype(np.uint8) * 255)

# ----------------------------
# MAIN
# ----------------------------
label_img = read_as_bgr(img_label)
clean_img = read_as_bgr(img_nolabel)

# Crop to common size
H = min(label_img.shape[0], clean_img.shape[0])
W = min(label_img.shape[1], clean_img.shape[1])
label_img = label_img[:H, :W]
clean_img = clean_img[:H, :W]

# Build mask using quantized palette (consistent with your hex extraction)
yellow_mask = build_mask_from_quantized_palette(label_img, TARGET_HEXES, BIN)

# Save a NON-overwriting preview
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
mask_name = f"yellow_mask_preview_{stamp}.png"
cv2.imwrite(mask_name, yellow_mask)
print(f"Wrote {mask_name} (white = detected yellow)")

base = Path(img_nolabel).stem.replace("-nolabel", "")

tiles = 0
yellow1 = 0
yellow0 = 0

for y in range(0, H - TILE + 1, TILE):
    for x in range(0, W - TILE + 1, TILE):
        label_tile = label_img[y:y+TILE, x:x+TILE]
        clean_tile = clean_img[y:y+TILE, x:x+TILE]
        tile_mask  = yellow_mask[y:y+TILE, x:x+TILE]

        match_pixels = int((tile_mask > 0).sum())
        has_yellow = int(match_pixels >= MIN_MATCH_PIXELS)

        fname = f"{base}_x{x:05d}_y{y:05d}_yellow{has_yellow}.png"
        cv2.imwrite(str(train_path / fname), label_tile)
        cv2.imwrite(str(test_path  / fname), clean_tile)

        tiles += 1
        yellow1 += has_yellow
        yellow0 += (1 - has_yellow)

print(f"Saved {tiles} tiles to train/ and test/")
print(f"Counts: yellow1={yellow1}, yellow0={yellow0}")
print(f"Params: BIN={BIN}, MIN_MATCH_PIXELS={MIN_MATCH_PIXELS}")


Wrote yellow_mask_preview_20260204_162041.png (white = detected yellow)
Saved 432 tiles to train/ and test/
Counts: yellow1=55, yellow0=377
Params: BIN=8, MIN_MATCH_PIXELS=3
